# 04 · Unsupervised Learning: Data Structure and Anomaly Detection

Unsupervised methods reveal structure in the data **without using F₂ as a label**.
We apply:

| Method | Purpose |
|---|---|
| **PCA** | Linear dimensionality reduction; variance explained |
| **t-SNE** | Non-linear embedding; visual cluster discovery |
| **K-Means** | Partition the kinematic $(x, Q^2)$ plane into regions |
| **DBSCAN** | Density-based clustering; finds irregular shapes |
| **Isolation Forest** | Anomaly / outlier detection across experiments |


In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams["figure.dpi"] = 120

from data_loader import load_lepton_dis, add_log_features, split_data, get_Xy, low_x_grid
from visualization import plot_coverage, plot_F2_vs_x, comparison_figure

DATA_DIR = "/Users/dikgarg/Desktop/Research/Neutrinos/postdoc/2025/ML/Data/Exp_data/LeptonDIS"

df_raw = load_lepton_dis(DATA_DIR)
df     = add_log_features(df_raw)
df_train, df_test = split_data(df, test_size=0.2, seed=42)

X_train, y_train = get_Xy(df_train)
X_test,  y_test  = get_Xy(df_test)

print(f"Training points: {len(X_train)}  |  Test points: {len(X_test)}")


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import IsolationForest
import seaborn as sns

# Feature matrix for unsupervised: (log10_x, log10_Q2, F2_norm)
scaler_u  = StandardScaler()
F2_norm   = (df["F2"].values - df["F2"].mean()) / df["F2"].std()
X_unsup   = np.column_stack([df["log10_x"].values,
                              df["log10_Q2"].values,
                              F2_norm])
X_unsup_s = scaler_u.fit_transform(X_unsup)


## Principal Component Analysis

In [ ]:
pca = PCA(n_components=3)
Z_pca = pca.fit_transform(X_unsup_s)

print("Explained variance ratio:")
for i, ev in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {ev:.3f}  ({ev*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sc = axes[0].scatter(Z_pca[:, 0], Z_pca[:, 1],
                     c=df["F2"].values, cmap="plasma",
                     s=10, alpha=0.7)
plt.colorbar(sc, ax=axes[0], label=r"$F_2^p$")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].set_title("PCA — coloured by F2")

sc2 = axes[1].scatter(Z_pca[:, 0], Z_pca[:, 1],
                      c=df["log10_x"].values, cmap="viridis",
                      s=10, alpha=0.7)
plt.colorbar(sc2, ax=axes[1], label=r"$\log_{10}(x)$")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[1].set_title(r"PCA — coloured by $\log_{10}(x)$")

plt.tight_layout()
plt.savefig("../results/figures/04_pca.png", dpi=150)
plt.show()


## t-SNE Embedding

In [ ]:
tsne = TSNE(n_components=2, perplexity=40, random_state=42, n_iter=1000)
Z_tsne = tsne.fit_transform(X_unsup_s)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sc = axes[0].scatter(Z_tsne[:, 0], Z_tsne[:, 1],
                     c=df["F2"].values, cmap="plasma", s=8, alpha=0.7)
plt.colorbar(sc, ax=axes[0], label=r"$F_2^p$")
axes[0].set_title("t-SNE — coloured by F2")
axes[0].set_xlabel("t-SNE 1"); axes[0].set_ylabel("t-SNE 2")

sc2 = axes[1].scatter(Z_tsne[:, 0], Z_tsne[:, 1],
                      c=df["log10_x"].values, cmap="viridis", s=8, alpha=0.7)
plt.colorbar(sc2, ax=axes[1], label=r"$\log_{10}(x)$")
axes[1].set_title(r"t-SNE — coloured by $\log_{10}(x)$")
axes[1].set_xlabel("t-SNE 1"); axes[1].set_ylabel("t-SNE 2")

plt.tight_layout()
plt.savefig("../results/figures/04_tsne.png", dpi=150)
plt.show()


## K-Means: Kinematic Region Clustering

In [ ]:
# Cluster only in the (log10_x, log10_Q2) plane for interpretability
X_kin_s = StandardScaler().fit_transform(
    df[["log10_x", "log10_Q2"]].values
)

inertias = []
K_range  = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_kin_s)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(list(K_range), inertias, "o-", color="#377eb8")
ax.set_xlabel("Number of clusters k"); ax.set_ylabel("Inertia")
ax.set_title("Elbow plot for K-Means")
ax.grid(True, ls="--", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
K_BEST = 5
km_best = KMeans(n_clusters=K_BEST, random_state=42, n_init=10)
labels_km = km_best.fit_predict(X_kin_s)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cmap_k = plt.get_cmap("tab10", K_BEST)
axes[0].scatter(df["x"], df["Q2"], c=labels_km, cmap=cmap_k,
                s=12, alpha=0.8)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel(r"$x$"); axes[0].set_ylabel(r"$Q^2$ [GeV$^2$]")
axes[0].set_title(f"K-Means clusters (k={K_BEST}) in $(x, Q^2)$ plane")

for k in range(K_BEST):
    mask = labels_km == k
    axes[1].scatter(df["x"].values[mask], df["F2"].values[mask],
                    s=8, alpha=0.7, label=f"Cluster {k}", color=cmap_k(k))
axes[1].set_xscale("log")
axes[1].set_xlabel(r"$x$"); axes[1].set_ylabel(r"$F_2^p$")
axes[1].set_title("F2 coloured by kinematic cluster")
axes[1].legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig("../results/figures/04_kmeans.png", dpi=150)
plt.show()

# Mean F2 per cluster
df_km = df.copy()
df_km["cluster"] = labels_km
print(df_km.groupby("cluster")[["x", "Q2", "F2"]].mean().round(4).to_string())


## DBSCAN: Density-Based Clustering

In [ ]:
dbscan = DBSCAN(eps=0.25, min_samples=5)
labels_db = dbscan.fit_predict(X_kin_s)

n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise    = (labels_db == -1).sum()
print(f"DBSCAN found {n_clusters} clusters, {n_noise} noise points")

fig, ax = plt.subplots(figsize=(7, 5))
unique_labels = sorted(set(labels_db))
cmap_d = plt.get_cmap("tab10", max(unique_labels) + 1)
for lbl in unique_labels:
    mask  = labels_db == lbl
    color = "grey" if lbl == -1 else cmap_d(lbl)
    name  = "Noise" if lbl == -1 else f"Cluster {lbl}"
    ax.scatter(df["x"].values[mask], df["Q2"].values[mask],
               s=12, alpha=0.8, color=color, label=name)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(r"$x$"); ax.set_ylabel(r"$Q^2$ [GeV$^2$]")
ax.set_title("DBSCAN clusters")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig("../results/figures/04_dbscan.png", dpi=150)
plt.show()


## Isolation Forest: Anomaly Detection

In [ ]:
# Detect outliers in the (log10_x, log10_Q2, F2) space
iso = IsolationForest(n_estimators=300, contamination=0.05, random_state=42)
anomaly_labels = iso.fit_predict(X_unsup_s)   # -1 = anomaly, +1 = normal

n_anomalies = (anomaly_labels == -1).sum()
print(f"Isolation Forest flagged {n_anomalies} anomalous points "
      f"({n_anomalies/len(df)*100:.1f}% of data)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = np.where(anomaly_labels == -1, "red", "steelblue")
alpha  = np.where(anomaly_labels == -1, 1.0, 0.3)

axes[0].scatter(df["x"], df["Q2"], c=colors, s=12, alpha=0.6)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel(r"$x$"); axes[0].set_ylabel(r"$Q^2$ [GeV$^2$]")
axes[0].set_title("Anomalies in $(x, Q^2)$ plane (red = flagged)")

axes[1].scatter(df["x"], df["F2"], c=colors, s=12, alpha=0.6)
axes[1].set_xscale("log")
axes[1].set_xlabel(r"$x$"); axes[1].set_ylabel(r"$F_2^p$")
axes[1].set_title("Anomalies in $(x, F_2)$ plane")

plt.tight_layout()
plt.savefig("../results/figures/04_isolation_forest.png", dpi=150)
plt.show()

# Which experiments do the anomalies come from?
df_anom = df.copy()
df_anom["anomaly"] = anomaly_labels == -1
print(df_anom.groupby("experiment")["anomaly"].sum().astype(int))
